# 07 Execution manifest (dry-run only)

Build rollback-ready manifests from the latest dry-run planner output.

This notebook still makes **zero file changes**. It only separates rows into:
- executable manifest
- keep register
- review queue
- blocked rows
- rollback manifest


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
PLAN_PATH = None  # set explicitly if you want; otherwise latest plan_dry_run_*.parquet is used

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUTS_DIR  =', OUTPUTS_DIR)


PROJECT_ROOT = c:\00_Developement\sch-file-organizer
OUTPUTS_DIR  = c:\00_Developement\sch-file-organizer\data\outputs


In [ ]:
from src.reporting import find_latest_output
from src.executor import ManifestConfig, build_execution_bundle, save_manifest_bundle, manifest_summary

if PLAN_PATH is None:
    PLAN_PATH = find_latest_output(OUTPUTS_DIR, 'plan_dry_run_')

print('PLAN_PATH =', PLAN_PATH)
plan = pd.read_parquet(PLAN_PATH)
print('Rows:', len(plan))
preview_cols = [c for c in ['relative_path', 'planner_action', 'planner_ready', 'planner_needs_user_input', 'planner_target_relative_path'] if c in plan.columns]
display(plan[preview_cols].head(15))


PLAN_PATH = c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260308_150102.parquet
Rows: 1806


,relative_path,planner_action,planner_ready,planner_needs_user_input,planner_target_relative_path
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
5,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
6,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
7,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
8,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...
9,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,True,False,_DUPLICATED/01_selected\random_1_dedup_target/...


## Manifest settings

Leave these conservative at first. In particular, keep collision blocking enabled.


In [3]:
EXECUTABLE_ACTIONS = (
    'move_to_policy_folder',
    'move_to_superseded_folder',
    'move_to_duplicate_folder',
    'move_to_deprecated_folder',
)
INCLUDE_KEEP_REGISTER = True
BLOCK_TARGET_COLLISIONS = True
REQUIRE_TARGET_PATH = True
REQUIRE_CHANGED_PATH = True

config = ManifestConfig(
    executable_actions=EXECUTABLE_ACTIONS,
    include_keep_register=INCLUDE_KEEP_REGISTER,
    block_target_collisions=BLOCK_TARGET_COLLISIONS,
    require_target_path=REQUIRE_TARGET_PATH,
    require_changed_path=REQUIRE_CHANGED_PATH,
)
config


ManifestConfig(executable_actions=('move_to_policy_folder', 'move_to_superseded_folder', 'move_to_duplicate_folder', 'move_to_deprecated_folder'), include_keep_register=True, block_target_collisions=True, require_target_path=True, require_changed_path=True)

In [4]:
bundle = build_execution_bundle(plan, config=config)
summary = manifest_summary(bundle)
summary


{'executable_rows': 130,
 'keep_rows': 0,
 'review_rows': 1676,
 'blocked_rows': 0,
 'rollback_rows': 130}

In [5]:
def _show(df, cols, n=20):
    safe_cols = [c for c in cols if c in df.columns]
    display(df[safe_cols].head(n))

_show(bundle.executable_manifest, ['relative_path', 'planner_action', 'execution_target_relative_path'])
_show(bundle.keep_register, ['relative_path', 'planner_action', 'keep_status'])
_show(bundle.blocked_manifest, ['relative_path', 'planner_action', 'execution_block_reason'])
_show(bundle.review_queue, ['relative_path', 'planner_action', 'planner_reason'])


,relative_path,planner_action,execution_target_relative_path
0,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
1,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
2,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
3,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
4,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
5,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
6,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
7,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
8,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...
9,01_selected\random_1_dedup_target\doclaynet_pd...,move_to_duplicate_folder,_DUPLICATED/01_selected\random_1_dedup_target/...


,relative_path,planner_action


,relative_path,planner_action,execution_block_reason


,relative_path,planner_action,planner_reason
0,01_selected\New Text Document.ogb,manual_review,needs classification or rename mapping
1,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,manual_review,needs classification or rename mapping
2,01_selected\doclaynet_pdf\doclaynet_pdf_0001_r...,manual_review,needs classification or rename mapping
3,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,manual_review,needs classification or rename mapping
4,01_selected\doclaynet_pdf\doclaynet_pdf_0002_p...,manual_review,needs classification or rename mapping
5,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,manual_review,needs classification or rename mapping
6,01_selected\doclaynet_pdf\doclaynet_pdf_0003_1...,manual_review,needs classification or rename mapping
7,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,manual_review,needs classification or rename mapping
8,01_selected\doclaynet_pdf\doclaynet_pdf_0004_N...,manual_review,needs classification or rename mapping
9,01_selected\doclaynet_pdf\doclaynet_pdf_0005_N...,manual_review,needs classification or rename mapping


In [6]:
STAMP = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
saved = save_manifest_bundle(bundle, OUTPUTS_DIR, stem=STAMP)
saved


{'executable_manifest': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_ready_20260308_150825.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_ready_20260308_150825.parquet')),
 'keep_register': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_keep_20260308_150825.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_keep_20260308_150825.parquet')),
 'review_queue': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_review_20260308_150825.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_review_20260308_150825.parquet')),
 'blocked_manifest': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_blocked_20260308_150825.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_blocked_20260308_150825.